## 2. B) Limpieza y Modelado de Ruido

Modelo de observacion: `x_obs = x_true + epsilon`, con hipotesis `epsilon ~ N(0, sigma^2)`.

In [13]:
import pandas as pd
import numpy as np
import json, re
from scipy import stats

with open("data/columnas_excluidas.json", encoding="utf-8") as f:
    excluidas = json.load(f)

df = pd.read_parquet("data/arrhythmia_preprocesado.parquet")
assert df.shape == (452, 262), f"shape inesperado: {df.shape}"
reap = [c for c in excluidas if c in df.columns]
assert not reap, f"columnas excluidas reaparecen en parquet: {reap}"

D = df.drop(columns="Class")

mask_cc = D.notna().all(axis=1)
y = df.loc[mask_cc, "Class"].to_numpy()
D = D.loc[mask_cc]

zero_var = [c for c in D.columns if D[c].std() <= 1e-12]
print(f"filas complete-case: {len(D)} de {len(df)} | std=0 en submuestra ({len(zero_var)}): {zero_var}")

Dn = D.drop(columns=zero_var)
X0 = Dn.to_numpy(dtype=float)
Xz = (X0 - X0.mean(axis=0)) / X0.std(axis=0)
n, d = Xz.shape
print(f"Matriz estandarizada: {n} x {d}")


filas complete-case: 420 de 452 | std=0 en submuestra (3): ['DI_ragged_R', 'DIII_JJ_ampl', 'AVR_R_prime_width']
Matriz estandarizada: 420 x 258


In [14]:
def familia(c):
    suf = re.sub(r'^(DI|DII|DIII|AVR|AVL|AVF|V[1-6])_', '', c)
    if suf.endswith('_ampl') or suf in ('QRSA', 'QRSTA'):
        return 'amplitud/area'
    if suf.endswith('_width') or suf.endswith('_interval') or 'duration' in suf or suf.endswith('_vector'):
        return 'duracion/angulo'
    if suf.startswith(('ragged_', 'diphasic_')):
        return 'morfologia'
    if suf == 'n_intrinsic_deflections':
        return 'conteo'
    return 'clinico'

fam = np.array([familia(c) for c in Dn.columns])

skew = stats.skew(X0, axis=0)
kurt = stats.kurtosis(X0, axis=0)
aprox_normal = (np.abs(skew) <= 2) & (kurt <= 5)

print("{:<18}{:>5}{:>9}{:>8}{:>8}".format('familia', 'cols', '~normal', 'skew>2', 'kurt>5'))
for fi in sorted(np.unique(fam)):
    m = fam == fi
    print("{:<18}{:>5}{:>9}{:>8}{:>8}".format(
        fi, int(m.sum()), int(aprox_normal[m].sum()),
        int((np.abs(skew[m]) > 2).sum()), int((kurt[m] > 5).sum())))

na = int(aprox_normal.sum())
print(f"Total: {d} columnas | {na} ({100*na/d:.0f}%) aproximadamente normales")
print("Mayor |skew|:", ', '.join(Dn.columns[np.argsort(np.abs(skew))[-6:]]))
print("Mayor kurtosis:", ', '.join(Dn.columns[np.argsort(kurt)[-6:]]))

mad = np.median(np.abs(Xz), axis=0)
sigma_ratio = 1.4826 * mad  # ratio robusto/clasico, NO sigma del ruido epsilon
print()
print("Escala robusta del ruido (sigma_robust = 1.4826 * MAD sobre Z-scala)")
print(f"  mediana {np.median(sigma_ratio):.3f} | rango [{sigma_ratio.min():.3f}, {sigma_ratio.max():.3f}]")
print(f"  cola mas pesada que gaussiana (sigma > 1.3): {(sigma_ratio > 1.3).sum()}")
print(f"  cola mas ligera que gaussiana (sigma < 0.7): {(sigma_ratio < 0.7).sum()}")
for fi in sorted(np.unique(fam)):
    m = fam == fi
    print(f"  {fi:<18} mediana sigma {np.median(sigma_ratio[m]):.3f}")

familia            cols  ~normal  skew>2  kurt>5
amplitud/area       112       32      70      78
clinico               5        2       1       3
conteo               12        2       7      10
duracion/angulo      65       21      34      43
morfologia           64       18      42      45
Total: 258 columnas | 75 (29%) aproximadamente normales
Mayor |skew|: AVF_S_prime_width, AVF_T_ampl, V1_ragged_R, V1_diphasic_R, DIII_Q_ampl, DI_QRSA
Mayor kurtosis: DII_Q_ampl, DI_diphasic_P, DIII_S_ampl, DII_ragged_T, AVF_S_prime_width, AVR_n_intrinsic_deflections

Escala robusta del ruido (sigma_robust = 1.4826 * MAD sobre Z-scala)
  mediana 0.614 | rango [0.072, 1.670]
  cola mas pesada que gaussiana (sigma > 1.3): 5
  cola mas ligera que gaussiana (sigma < 0.7): 151
  amplitud/area      mediana sigma 0.604
  clinico            mediana sigma 0.848
  conteo             mediana sigma 0.598
  duracion/angulo    mediana sigma 0.646
  morfologia         mediana sigma 0.527


In [20]:
mn = X0.min(axis=0)
has_neg = mn < 0
has_zero = (mn == 0) & (~has_neg)

Xlog1p = np.full_like(X0, np.nan)
for j in range(X0.shape[1]):
    if not has_neg[j]:
        Xlog1p[:, j] = np.log1p(X0[:, j])

sk_orig = stats.skew(X0[:, ~has_neg], axis=0)
ku_orig = stats.kurtosis(X0[:, ~has_neg], axis=0)
sk_log = stats.skew(Xlog1p[:, ~has_neg], axis=0)
ku_log = stats.kurtosis(Xlog1p[:, ~has_neg], axis=0)

print("viabilidad logaritmic")
print(f"negativos (log no aplica): {int(has_neg.sum())} | min=0 (log1p): {int(has_zero.sum())} | >0: {int((~has_neg & ~has_zero).sum())}")

print()
print("{:<18}{:>6}{:>11}{:>11}{:>9}".format('familia', 'cols', 'sk mejora', 'kurt mejora', 'ok_nuevo'))
fam_v = fam[~has_neg]
for fi in sorted(np.unique(fam_v)):
    m = fam_v == fi
    if m.sum() == 0:
        continue
    print("{:<18}{:>6}{:>11}{:>11}{:>9}".format(
        fi, int(m.sum()),
        int((np.abs(sk_log[m]) < np.abs(sk_orig[m])).sum()),
        int((ku_log[m] < ku_orig[m]).sum()),
        int(((np.abs(sk_log[m]) <= 2) & (ku_log[m] <= 5)).sum())))

ok_orig = int(((np.abs(sk_orig) <= 2) & (ku_orig <= 5)).sum())
ok_log = int(((np.abs(sk_log) <= 2) & (ku_log <= 5)).sum())
print(f"aprox-normales entre aplicables: {ok_log}")

viabilidad logaritmic
negativos (log no aplica): 96 | min=0 (log1p): 156 | >0: 6

familia             cols  sk mejorakurt mejora ok_nuevo
amplitud/area         67         35         35       14
clinico                5          3          3        2
conteo                 7          4          3        0
duracion/angulo       42         24         24       12
morfologia            41         24         21       10
aprox-normales entre aplicables: 38



# Distribución de las variables

- De las **258 variables**, solo **75 (29%)** cumplen el criterio de distribución aproximadamente normal.
- La mayoría presenta **asimetría elevada y/o colas pesadas**
- La escala tiene una mediana de **0.619**, y **150 variables** presentan una escala < 0.7.
- Solo **5 variables** presentan colas particularmente pesadas (> 1.3).

### Transformación logarítmica

- `log1p` es aplicable a **162 variables**; las otras **96 contienen valores negativos**.
- La transformación mejora la asimetría en **90 variables** y la kurtosis en **86**.
- Sin embargo, las variables aproximadamente normales pasan de **39 a 38**, por lo que **no existe una mejora global de normalidad**.
- Por tanto, **no se justifica aplicar `log1p` de forma general**; puede utilizarse selectivamente en variables donde produzca una mejora clara.


### 4.2.2 Deteccion de outliers: Z-score (L2) y Distancia de Mahalanobis

In [21]:
zabs = np.abs(Xz)
nvars = (zabs > 3).sum(axis=1)

# Correccion por multiplicidad: Bonferroni con d=258 tests por fila (2 colas)
from scipy.stats import norm
c_bon = float(norm.ppf(1 - 0.05 / (2 * d)))
nvars_bon = (zabs > c_bon).sum(axis=1)

print(" Outliers por Z-score (L2) ")
print("{:<14}{:>8}".format('umbral |z|', 'filas'))
for t in (3, 4, 5):
    print("{:<14}{:>8}".format(f"> {t}", int((zabs.max(axis=1) > t).sum())))

print(f"filas con >=1 variable |z|>3: {(nvars > 0).sum()} | con >=10 variables: {(nvars >= 10).sum()}")
print(f"\nBonferroni (d={d} tests por fila, 5% por fila, 2 colas): |z| > {c_bon:.2f}")
print(f"filas con >=1 variable |z|>{c_bon:.2f}: {(nvars_bon > 0).sum()}  (el crudo |z|>3 marcaba {(nvars > 0).sum()})")

col_act_bon = (zabs > c_bon).sum(axis=0)
top = np.argsort(col_act_bon)[::-1][:8]
print("\nvariables mas activas (veces |z|>" + f"{c_bon:.2f}" + ", Bonferroni):")
for i in top:
    print(f"  {Dn.columns[i]:<30} x{col_act_bon[i]}")


 Outliers por Z-score (L2) 
umbral |z|       filas
> 3                244
> 4                195
> 5                166
filas con >=1 variable |z|>3: 244 | con >=10 variables: 36

Bonferroni (d=258 tests por fila, 5% por fila, 2 colas): |z| > 3.73
filas con >=1 variable |z|>3.73: 203  (el crudo |z|>3 marcaba 244)

variables mas activas (veces |z|>3.73, Bonferroni):
  AVL_ragged_P                   x15
  DII_diphasic_P                 x14
  V5_P_ampl                      x12
  DII_n_intrinsic_deflections    x11
  AVR_R_width                    x11
  AVF_R_width                    x11
  V3_T_ampl                      x10
  V3_S_width                     x10


In [23]:
from sklearn.covariance import LedoitWolf

mu = Xz.mean(axis=0)

def mahal(S, nu, etiqueta):
    Si = np.linalg.pinv(S)
    d2 = np.sum((Xz - mu) @ Si * (Xz - mu), axis=1)
    dm = np.sqrt(d2)
    print(f"Mahalanobis ({etiqueta})")
    print(f"DM rango: [{dm.min():.2f}, {dm.max():.2f}]")
    for a in (0.975, 0.99, 0.999):
        thr = np.sqrt(stats.chi2.ppf(a, nu))
        print(f"DM > sqrt(chi2_{a:.3f}({nu})) = {thr:.2f}  ->  {(dm > thr).sum()} outliers")
    return dm

S_muestral = np.cov(Xz, rowvar=False)
nu = np.linalg.matrix_rank(S_muestral)
print(f"rank(Sigma muestral) = {nu} de {d} (singular)")
dm_clasica = mahal(S_muestral, nu, "Sigma muestral, pseudo-inversa [referencia]")

lw = LedoitWolf().fit(Xz)
S_lw = lw.covariance_
nu_lw = d
print(f"\nLedoit-Wolf: shrinkage = {lw.shrinkage_:.3f} | cond(S_lw) = {np.linalg.cond(S_lw):.2e}")
dm = mahal(S_lw, nu_lw, "Ledoit-Wolf [principal]")

orden = np.argsort(dm)[::-1]
mask = np.ones(n, dtype=bool)
mask[orden[:20]] = False
mu_r = Xz[mask].mean(axis=0)
S_r = LedoitWolf().fit(Xz[mask]).covariance_
dm_r = mahal(S_r, nu_lw, "recomputada sin top-20")

new_top20 = np.argsort(dm_r)[::-1][:20]
solapamiento = len(set(orden[:20]) & set(new_top20))
print(f"\nSensibilidad (masking): solapamiento entre top-20 original y nuevo: {solapamiento}/20")

rank(Sigma muestral) = 252 de 258 (singular)
Mahalanobis (Sigma muestral, pseudo-inversa [referencia])
DM rango: [9.66, 20.45]
DM > sqrt(chi2_0.975(252)) = 17.26  ->  128 outliers
DM > sqrt(chi2_0.990(252)) = 17.53  ->  118 outliers
DM > sqrt(chi2_0.999(252)) = 18.09  ->  107 outliers

Ledoit-Wolf: shrinkage = 0.308 | cond(S_lw) = 4.85e+01
Mahalanobis (Ledoit-Wolf [principal])
DM rango: [4.92, 23.55]
DM > sqrt(chi2_0.975(258)) = 17.45  ->  48 outliers
DM > sqrt(chi2_0.990(258)) = 17.71  ->  43 outliers
DM > sqrt(chi2_0.999(258)) = 18.27  ->  39 outliers
Mahalanobis (recomputada sin top-20)
DM rango: [5.48, 116.72]
DM > sqrt(chi2_0.975(258)) = 17.45  ->  56 outliers
DM > sqrt(chi2_0.990(258)) = 17.71  ->  54 outliers
DM > sqrt(chi2_0.999(258)) = 18.27  ->  49 outliers

Sensibilidad (masking): solapamiento entre top-20 original y nuevo: 20/20


In [18]:
thr = np.sqrt(stats.chi2.ppf(0.975, nu_lw))
o = dm > thr
print(f"=== Perfil de los {int(o.sum())} outliers (DM Ledoit-Wolf, 97.5%) ===")

uu, cu = np.unique(y[o], return_counts=True)
tot = pd.Series(y).value_counts()
print("concentracion por clase (% de la clase):")
for u, c in zip(uu, cu):
    print(f"  clase {u}: {c}/{int(tot[u])} ({100*c/int(tot[u]):.0f}%)")

nvo = nvars_bon[o]
print(f"univariados Bonferroni (>=1 var |z|>{c_bon:.2f}): {(nvo > 0).sum()} | solo multivariados: {(nvo == 0).sum()}")

print("Para cada familia, cuantos outliers tienen >=1 variable de esa familia con |z|>3:")
for fi in sorted(np.unique(fam)):
    activo = (zabs[o][:, fam == fi] > 3).any(axis=1)
    if activo.sum() > 0:
        print(f"  {fi:<18} {int(activo.sum())} filas")


=== Perfil de los 48 outliers (DM Ledoit-Wolf, 97.5%) ===
concentracion por clase (% de la clase):
  clase 1: 12/237 (5%)
  clase 2: 8/36 (22%)
  clase 3: 3/13 (23%)
  clase 4: 2/14 (14%)
  clase 5: 4/13 (31%)
  clase 7: 1/2 (50%)
  clase 9: 5/9 (56%)
  clase 10: 10/48 (21%)
  clase 14: 1/4 (25%)
  clase 16: 2/18 (11%)
univariados Bonferroni (>=1 var |z|>3.73): 48 | solo multivariados: 0
Para cada familia, cuantos outliers tienen >=1 variable de esa familia con |z|>3:
  amplitud/area      43 filas
  clinico            5 filas
  conteo             21 filas
  duracion/angulo    44 filas
  morfologia         41 filas


### 4.2.2 - Conclusion analisis outleirs

- El análisis por **Z-score** detectó una presencia importante de valores extremos:
  - **244/420 filas** tienen al menos un `|z| > 3`, lo que indica una presencia considerable de observaciones potencialmente atípicas.
  - Con **Bonferroni**, todavía quedan **203/420 filas**, confirmando que una proporción relevante de los casos presenta valores extremos más allá de lo esperable por azar.
  - Algunas variables concentran más valores extremos, especialmente `AVL_ragged_P` y `DII_diphasic_P`, por lo que podrían estar contribuyendo en mayor medida a la detección univariada.

- La distancia de **Mahalanobis clásica** no es adecuada como referencia principal porque la covarianza es singular (**rango 252/258**), generando entre **107 y 128 outliers** según el umbral. Esto limita la confiabilidad de esta estimación.

- Con **Ledoit-Wolf**, la covarianza queda bien condicionada (`cond ≈ 48.5`) y se detectan entre **39 y 48 outliers**, dependiendo del nivel de corte, proporcionando una detección multivariada más estable.

- Al eliminar los **20 casos más extremos** y recalcular, se detectan entre **49 y 56 outliers**. El **top-20 se mantiene exactamente (20/20)**, lo que indica una alta estabilidad de los casos identificados como más extremos.



### 4.2.3 Por que DM supera a la distancia euclidea
### Demostración matemática

Partimos de la distancia Euclídea:

$$
D_E(x)=\sqrt{(x-\mu)^T(x-\mu)}
$$

Esta distancia supone que todas las variables tienen varianza 1 y que no están correlacionadas. En términos de covarianza, esto equivale a:

$$
\Sigma=I
$$

Por lo tanto:

$$
D_E^2(x)
=
(x-\mu)^TI^{-1}(x-\mu)
$$

Como:

$$
I^{-1}=I
$$

tenemos:

$$
D_E^2(x)
=
(x-\mu)^T(x-\mu)
$$

Ahora reemplazamos $I$ por la matriz de covarianzas $\Sigma$:

$$
D_M^2(x)
=
(x-\mu)^T\Sigma^{-1}(x-\mu)
$$

Por tanto:

$$
\boxed{
D_M(x)=\sqrt{(x-\mu)^T\Sigma^{-1}(x-\mu)}
}
$$

Para demostrar qué efecto tiene $\Sigma^{-1}$, consideremos primero el caso de variables no correlacionadas:

$$
\Sigma=
\begin{pmatrix}
\sigma_1^2 & 0\\
0 & \sigma_2^2
\end{pmatrix}
$$

Entonces:

$$
\Sigma^{-1}=
\begin{pmatrix}
\frac{1}{\sigma_1^2} & 0\\
0 & \frac{1}{\sigma_2^2}
\end{pmatrix}
$$

y:

$$
D_M^2(x)
=
\frac{(x_1-\mu_1)^2}{\sigma_1^2}
+
\frac{(x_2-\mu_2)^2}{\sigma_2^2}
$$

Es decir:

$$
D_M^2(x)
=
\left(\frac{x_1-\mu_1}{\sigma_1}\right)^2
+
\left(\frac{x_2-\mu_2}{\sigma_2}\right)^2
$$

Por lo tanto, Mahalanobis equivale a calcular una **distancia Euclídea después de estandarizar cada variable por su desviación estándar**.

Cuando las variables están correlacionadas, $\Sigma$ además contiene las covarianzas:

$$
\Sigma=
\begin{pmatrix}
\sigma_1^2 & \sigma_{12}\\
\sigma_{12} & \sigma_2^2
\end{pmatrix}
$$

y $\Sigma^{-1}$ incorpora estas correlaciones en el cálculo de la distancia.

Así, matemáticamente:

$$
\boxed{
\text{Euclídea}
=
\text{Mahalanobis cuando } \Sigma=I
}
$$

y:

$$
\boxed{
\text{Mahalanobis}
=
\text{distancia Euclídea ajustada por varianzas y covarianzas}
}
$$

En este dataset, como la covarianza clásica es singular, se utiliza $\Sigma_{LW}$ de Ledoit-Wolf:

$$
\boxed{
D_{LW}(x)
=
\sqrt{(x-\mu)^T\Sigma_{LW}^{-1}(x-\mu)}
}
$$

lo que permite calcular una distancia multivariada estable para detectar observaciones alejadas de la estructura conjunta de los datos.
